# IA-SPA: Notebook 2 -- San Francisco End-to-End Workflow

**Interference-Aware Submodular Placement Algorithm**  
L. Taus, R. Tsai, and J. G. Andrews (2026)

> **Paper:** "Optimal Transmitter Placement in Realistic Urban Environments"  
> Submitted to *IEEE Transactions on Wireless Communications*  
> [arXiv:2604.28153 [cs.IT]](https://arxiv.org/abs/2604.28153)


---

This notebook walks through the full IA-SPA pipeline on Sionna's built-in
San Francisco 3-D scene, reproducing the AT&T and T-Mobile comparison from
Tables I and II of the paper.

**Prerequisites**
- GPU with >= 16 GB VRAM (A100 / H100 recommended)
- `sionna`, `ia_spa` installed (see `README.md`)
- Tower data files in `data/TowerData/SF_*.npy`

**Estimated runtime**  
Basis-function pre-computation: several hours (GPU).  
Optimisation loop: ~1 min per greedy iteration.

---

## Step 0 -- Imports and Configuration

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import yaml
from pathlib import Path
from sionna.rt import load_scene, scene as sionna_scenes, PlanarArray

from ia_spa import (
    TowerOptimizer, run_greedy,
    build_terrain_tree, apply_tower_height,
    evaluate_and_save, load_greedy_positions,
)

with open('../config/config.yaml') as fh:
    cfg = yaml.safe_load(fh)

BASIS_FOLDER   = Path(cfg['paths']['basis_functions_sf'])
TOWER_DATA     = Path(cfg['paths']['tower_data'])
RESULTS_FOLDER = Path(cfg['paths']['results']) / 'SF_max'
TOWER_HEIGHT   = float(cfg['transmitter']['tower_height_m'])
POWER_DBM      = float(cfg['transmitter']['power_dbm'])
FREQ_HZ        = float(cfg['scene']['frequency_hz'])
HEIGHTS        = cfg['receiver']['heights_m']
BW_HZ          = float(cfg['link']['bandwidth_hz'])
GAMMA          = float(cfg['link']['snr_gap_gamma'])
N_ITER         = int(cfg['optimizer']['n_iter'])

print('Configuration loaded.')
print(f'  Frequency : {FREQ_HZ/1e9:.2f} GHz')
print(f'  Bandwidth : {BW_HZ/1e6:.0f} MHz')
print(f'  TX power  : {POWER_DBM} dBm')
print(f'  Heights   : {HEIGHTS} m')
print(f'  Iterations: {N_ITER}')

## Step 1 -- Pre-compute Basis Functions

This is the expensive step.  Skip this cell if basis functions have already been computed.
To run it from the command line instead:
```bash
python scripts/compute_basis_sf.py --config config/config.yaml
```

In [ ]:
SKIP_IF_EXISTS = True   # <- Set to False to force recomputation

coord_file = BASIS_FOLDER / 'Sionna' / '0_Coordinates.txt'
if coord_file.exists() and SKIP_IF_EXISTS:
    print(f'Basis functions already exist at {BASIS_FOLDER}. Skipping.')
else:
    from ia_spa import compute_sdf, extract_candidate_locations, compute_basis_functions

    sdf_path = Path(cfg['paths']['map_data']) / 'SDF_SF.npy'

    # 1a. Build SDF
    if not sdf_path.exists():
        sf_sdf = load_scene(sionna_scenes.san_francisco, merge_shapes=False)
        compute_sdf(sf_sdf, sdf_path)

    # 1b. Extract candidate locations
    sf_cands = load_scene(sionna_scenes.san_francisco, merge_shapes=False)
    bbox = sf_cands._scene.bbox()
    sdf_grid = np.load(sdf_path)
    candidates = extract_candidate_locations(sf_cands, sdf_grid, bbox.min, bbox.max)
    print(f'Found {len(candidates)} candidate locations.')

    # 1c. Ray-trace each candidate
    sf_scene = load_scene(sionna_scenes.san_francisco, merge_shapes=True)
    compute_basis_functions(
        sf_scene, candidates, BASIS_FOLDER,
        power_dbm=POWER_DBM,
        bandwidth_hz=BW_HZ,
        snr_gap_gamma=GAMMA,
        frequency_hz=FREQ_HZ,
    )
    print('Basis functions computed.')

## Step 2 -- Run the Greedy Optimiser

In [ ]:
optimizer = TowerOptimizer(BASIS_FOLDER, aggregation='max')
print(f'Loaded {optimizer.n_candidates} candidate locations.')

run_greedy(optimizer, results_folder=RESULTS_FOLDER, n_iter=N_ITER)
print('Greedy optimisation complete.')

## Step 3 -- Evaluate Against Reference Deployments

In [ ]:
sf_scene = load_scene(sionna_scenes.san_francisco, merge_shapes=False)
sf_scene.tx_array = PlanarArray(num_rows=1, num_cols=1, pattern='iso', polarization='V')
sf_scene.frequency = FREQ_HZ

terrain_tree, terrain_verts = build_terrain_tree(sf_scene, 'Terrain')
save_dir = RESULTS_FOLDER.parent / (RESULTS_FOLDER.name + '_Processed')

# Reference deployments
refs = {
    'ATT_ref':         np.load(TOWER_DATA / 'SF_ATT.npy'),
    'TMobile_ref':     np.load(TOWER_DATA / 'SF_TMobile.npy'),
    'ATT_TMobile_ref': np.load(TOWER_DATA / 'SF_ATT_TMobile.npy'),
}
refs = {k: apply_tower_height(v, terrain_tree, terrain_verts, TOWER_HEIGHT) for k, v in refs.items()}

for name, pos in refs.items():
    evaluate_and_save(sf_scene, name, pos, save_dir, HEIGHTS, POWER_DBM, BW_HZ, GAMMA)

# Greedy results
greedy_pos = load_greedy_positions(RESULTS_FOLDER)
greedy_pos = apply_tower_height(greedy_pos, terrain_tree, terrain_verts, TOWER_HEIGHT)
for ref_name, ref_pos in refs.items():
    carrier = ref_name.replace('_ref', '')
    evaluate_and_save(
        sf_scene, f'{carrier}_Greedy',
        greedy_pos[:ref_pos.shape[0]], save_dir,
        HEIGHTS, POWER_DBM, BW_HZ, GAMMA,
    )

## Step 4 -- Visualise and Tabulate Results

In [ ]:
import pandas as pd

def summary_stats(rate_map):
    r = rate_map.ravel() / 1e6  # -> Mbps
    return {
        'Mean (Mbps)':    round(r.mean(), 2),
        'Std (Mbps)':     round(r.std(), 2),
        'Max (Mbps)':     round(r.max(), 2),
        '5th pct (Mbps)': round(np.percentile(r, 5), 2),
    }

rows = {}
for tag in ['ATT_ref', 'ATT_Greedy', 'TMobile_ref', 'TMobile_Greedy']:
    rate = np.load(save_dir / f'{tag}_rate.npy')
    rows[tag] = summary_stats(rate)

df = pd.DataFrame(rows).T
df.index.name = 'Scenario'
print(df.to_string())
df.to_csv(save_dir / 'summary_table.csv')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

pairs = [('ATT_ref', 'ATT_Greedy'), ('TMobile_ref', 'TMobile_Greedy')]
for row, (ref_tag, greedy_tag) in enumerate(pairs):
    ref_rate    = np.load(save_dir / f'{ref_tag}_rate.npy') / 1e6
    greedy_rate = np.load(save_dir / f'{greedy_tag}_rate.npy') / 1e6
    vmax = max(ref_rate.max(), greedy_rate.max())

    ax_ref = axes[row, 0]
    ax_opt = axes[row, 1]

    im0 = ax_ref.imshow(ref_rate, cmap='viridis', vmin=0, vmax=vmax, origin='lower')
    ax_ref.set_title(f'{ref_tag}\nMean = {ref_rate.mean():.1f} Mbps', fontsize=11)
    ax_ref.axis('off')
    fig.colorbar(im0, ax=ax_ref, shrink=0.8, label='Rate (Mbps)')

    im1 = ax_opt.imshow(greedy_rate, cmap='viridis', vmin=0, vmax=vmax, origin='lower')
    ax_opt.set_title(f'{greedy_tag}\nMean = {greedy_rate.mean():.1f} Mbps', fontsize=11)
    ax_opt.axis('off')
    fig.colorbar(im1, ax=ax_opt, shrink=0.8, label='Rate (Mbps)')

fig.suptitle('San Francisco: Reference vs. IA-SPA Throughput', fontsize=14, weight='bold')
plt.tight_layout()
plt.savefig(save_dir / 'sf_comparison.png', dpi=150, bbox_inches='tight')
plt.show()